In [29]:
from datasets import load_dataset

# Problem bei normalem Laden
dataset = load_dataset(
    "ptb_text_only",
    split="train",
    trust_remote_code=True
)

ds_split = dataset.train_test_split(test_size=0.2)
train_set = ds_split["train"]
test_set = ds_split["test"]

In [30]:
import regex as re
from collections import Counter
from typing import Dict, List, Tuple

class BPETokenizer:
    """A byte-level BPE tokenizer."""

    PAT = re.compile(
        r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""",
        re.UNICODE,
    )

    def __init__(self):
        # Initialise the base vocabulary: all 256 single-byte tokens.
        self.vocab: Dict[int, bytes] = {i: bytes([i]) for i in range(256)}
        self.merges: List[Tuple[bytes, bytes]] = []

    def train(self, corpus: str, vocab_size: int) -> None:
        """Learn BPE merges from *corpus* until len(self.vocab) == vocab_size."""
        # Step 1

        # Häufigkeiten der Bytefolgen im Korpus
        pre_token_counts: Dict[bytes, int] = Counter()
        for match in self.PAT.findall(corpus):
            pre_token_counts[match.encode("utf-8")] += 1

        # Step 2

        # Convert to tuples of single bytes
        pre_token_tuples: Dict[Tuple[bytes, ...], int] = {}
        for token_bytes, count in pre_token_counts.items():
            t = tuple(token_bytes[i:i+1] for i in range(len(token_bytes)))
            pre_token_tuples[t] = pre_token_tuples.get(t, 0) + count

        # Count adjacent pairs
        pair_freq: Dict[Tuple[bytes, bytes], int] = Counter()
        for token_tuple, count in pre_token_tuples.items():
            for i in range(len(token_tuple) - 1):
                pair_freq[(token_tuple[i], token_tuple[i+1])] += count

        # Training loop
        num_merges = vocab_size - 256

        for i in range(num_merges):
            # falls keine Paare mehr -> Abbruch
            if not pair_freq:
                break

            max_freq = max(pair_freq.values())
            # Kandidaten mit max_freq suchen
            candidates = [pair for pair, freq in pair_freq.items() if freq == max_freq]

            best_pair = max(candidates)
            # merge speichern
            self.merges.append(best_pair)
            # besten byte pairs zu neuem token
            merged_token = best_pair[0] + best_pair[1]
            # neus token in vocab speichern
            self.vocab[len(self.vocab)] = merged_token

            # neues Dict für pre tokens nach merge
            new_pre_token_tuples: Dict[Tuple[bytes, ...], int] = {}

            for token_tuple, count in pre_token_tuples.items():
                new_tuple = []
                i = 0
                # Wird neuere merge durchgeführt
                new_merge = False

                while i < len(token_tuple):
                    # ist aktuelles Element teil des best_pair, dann wird merge durchgeführt
                    if (i < len(token_tuple) - 1 and token_tuple[i] == best_pair[0] and token_tuple[i + 1] == best_pair[1]):
                        new_merge = True

                        # alte linke Nachbarn entfernen
                        if i > 0:
                            old_left_pair = (token_tuple[i - 1], token_tuple[i])
                            # Häufigkeiten des alten pairs entfernen
                            pair_freq[old_left_pair] -= count

                            if pair_freq[old_left_pair] == 0:
                                del pair_freq[old_left_pair]

                        # aktuelles pair entfernen
                        pair_freq[best_pair] -= count

                        if pair_freq[best_pair] == 0:
                            del pair_freq[best_pair]

                        # alte rechte Nachbarn entfernen
                        if i + 2 < len(token_tuple):
                            old_right_pair = (token_tuple[i + 1], token_tuple[i + 2])
                            pair_freq[old_right_pair] -= count

                            if pair_freq[old_right_pair] <= 0:
                                del pair_freq[old_right_pair]

                        # neue linke Nachbarn hinzufügen
                        if i > 0:
                            new_left_pair = (token_tuple[i - 1], merged_token)
                            # Häufigkeiten des neuen pairs hinzufügen
                            pair_freq[new_left_pair] += count

                        # neue rechte Nachbarn hinzufügen
                        if i + 2 < len(token_tuple):
                            new_right_pair = (merged_token, token_tuple[i + 2])
                            pair_freq[new_right_pair] += count

                        new_tuple.append(merged_token)
                        i += 2
                    # wenn kein neues token, dann altes übernehmen
                    else:
                        new_tuple.append(token_tuple[i])
                        i += 1
                # Liste zurück in tuple
                new_tuple = tuple(new_tuple)

                if new_merge == True:
                    new_pre_token_tuples[new_tuple] = (new_pre_token_tuples.get(new_tuple, 0) + count)

                else:
                    new_pre_token_tuples[token_tuple] = (new_pre_token_tuples.get(token_tuple, 0) + count)

            pre_token_tuples = new_pre_token_tuples

    # vorgegebene Hilfsfunktion: führt einzelnen merge pass durch
    @staticmethod
    def _apply_merge(parts: List[bytes], pair: Tuple[bytes, bytes]) -> List[bytes]:
        first, second = pair
        result, i = [], 0
        while i < len(parts):
            if i < len(parts) - 1 and parts[i] == first and parts[i+1] == second:
                result.append(first + second)
                i += 2
            else:
                result.append(parts[i])
                i += 1
        return result

    def encode(self, text: str) -> List[int]:
        """Encode *text* into a list of token IDs using the learned merges."""
        # vocab dict umdrehen: byte token: id
        vocab_reversed = {}
        for id, token in self.vocab.items():
            vocab_reversed[token] = id

        ids = []

        # pretokenize wie in train
        for match in self.PAT.findall(text):
            token_bytes = match.encode("utf-8")

            # pre token in einzelne bytes
            parts = [token_bytes[i:i + 1]for i in range(len(token_bytes))]

            # merges durchführen
            for pair in self.merges:
                parts = self._apply_merge(parts, pair)

            # byte token in id
            for byte in parts:
                ids.append(vocab_reversed[byte])

        return ids

    def decode(self, ids: List[int]) -> str:
        """Decode a list of token IDs back to a Unicode string."""
        raw = b"".join(self.vocab[i] for i in ids)
        return raw.decode("utf-8", errors="replace")

In [31]:
text = " ".join(train_set["sentence"])
tokenizer = BPETokenizer()
tokenizer.train(text, vocab_size=500)

print(len(tokenizer.merges))

244


In [32]:
print("First 10 merges:")
for merge in tokenizer.merges[:10]:
    print(merge, "merded to: ", merge[0] + merge[1])

print("Last 10 merges:")
for merge in tokenizer.merges[-10:]:
    print(merge, "merged to: ", merge[0] + merge[1])

First 10 merges:
(b' ', b't') merded to:  b' t'
(b' ', b'a') merded to:  b' a'
(b'i', b'n') merded to:  b'in'
(b' t', b'h') merded to:  b' th'
(b'e', b'r') merded to:  b'er'
(b' ', b's') merded to:  b' s'
(b' th', b'e') merded to:  b' the'
(b'u', b'n') merded to:  b'un'
(b'o', b'n') merded to:  b'on'
(b'r', b'e') merded to:  b're'
Last 10 merges:
(b'ac', b't') merged to:  b'act'
(b' n', b'ot') merged to:  b' not'
(b'an', b'k') merged to:  b'ank'
(b'an', b's') merged to:  b'ans'
(b' e', b'ar') merged to:  b' ear'
(b' mo', b're') merged to:  b' more'
(b'd', b'er') merged to:  b'der'
(b' re', b'p') merged to:  b' rep'
(b' s', b'ays') merged to:  b' says'
(b' d', b'o') merged to:  b' do'


In [33]:
sentence = "the cat sat"
ids = tokenizer.encode(sentence)

print("token ids:")
print(ids)

print("byte tokens:")
for token_id in ids:
    print(token_id, tokenizer.vocab[token_id])

token ids:
[334, 101, 266, 280, 261, 280]
byte tokens:
334 b'th'
101 b'e'
266 b' c'
280 b'at'
261 b' s'
280 b'at'


In [34]:
decoded = tokenizer.decode(ids)
print(decoded)

the cat sat


In [35]:
for sentence in test_set["sentence"][:20]:
    assert tokenizer.decode(tokenizer.encode(sentence)) == sentence

In [36]:
sentences = [
    "the company said it will offer the securities",
    "trading was halted in several stocks",
]
for s in sentences:
    ids = tokenizer.encode(s)
    tokens = [tokenizer.vocab[i] for i in ids]
    print(f"Text:   {s}")
    print(f"IDs:    {ids}")
    print(f"Tokens: {tokens}")
    assert tokenizer.decode(ids) == s, "Round-trip failed!"

Text:   the company said it will offer the securities
IDs:    [334, 101, 449, 354, 316, 413, 417, 260, 262, 261, 312, 314, 277, 385]
Tokens: [b'th', b'e', b' company', b' said', b' it', b' will', b' off', b'er', b' the', b' s', b'ec', b'ur', b'it', b'ies']
Text:   trading was halted in several stocks
IDs:    [454, 323, 291, 388, 287, 286, 116, 283, 279, 432, 345, 286, 469, 115]
Tokens: [b'tr', b'ad', b'ing', b' was', b' h', b'al', b't', b'ed', b' in', b' se', b'ver', b'al', b' stock', b's']


In [37]:
import numpy as np
compress_ratio = []

for sentence in test_set["sentence"]:
    raw_bytes = len(sentence.encode("utf-8"))
    bpe_tokens = len(tokenizer.encode(sentence))

    compress_ratio.append(raw_bytes / bpe_tokens)

compress_ratio = np.array(compress_ratio)

print("vocab_size = 500")
print("mean compression ratio:", compress_ratio.mean())
print("standard derivation compression ratio: ", compress_ratio.std())

vocab_size = 500
mean compression ratio: 2.190394993063529
standard derivation compression ratio:  0.19711952872971578


In [38]:
tokenizer_1000 = BPETokenizer()
tokenizer_1000.train(text, vocab_size=1000)

compress_ratio = []

for sentence in test_set["sentence"]:
    raw_bytes = len(sentence.encode("utf-8"))
    bpe_tokens = len(tokenizer_1000.encode(sentence))

    compress_ratio.append(raw_bytes / bpe_tokens)

compress_ratio = np.array(compress_ratio)

print("vocab_size = 1000")
print("mean compression ratio:", compress_ratio.mean())
print("standard derivation compression ratio: ", compress_ratio.std())

vocab_size = 1000
mean compression ratio: 2.8205921182092673
standard derivation compression ratio:  0.36473811159425257


In [39]:
longest_tokens = sorted(tokenizer_1000.vocab.items(),key=lambda item: len(item[1]),reverse=True)[:20]

for id, token in longest_tokens:
    print(token)

b' government'
b' securities'
b' investment'
b' president'
b' companies'
b' investors'
b' yesterday'
b' financial'
b' executive'
b' officials'
b' business'
b' interest'
b' official'
b' exchange'
b' earnings'
b' chairman'
b' american'
b' industry'
b' national'
b' analysts'
